# Chapter 4 (part 3) — Interpreting the Regression Equation

Covers the Ch 4 section on interpretation: Correlated Predictors → Multicollinearity → Confounding Variables → Interactions and Main Effects, on Ames Housing.

The thread: **a regression coefficient is a conditional statement — "the difference in y per unit of x among observations that look the same on everything else in the model" — so what's *in* the model changes what every coefficient means.** Correlated predictors make that conditional comparison data-starved (few houses differ on one variable but not the other). Confounding means a missing variable is silently doing the work. Interactions mean the answer to "what's a square foot worth?" depends on who you ask.

This is the notebook where regression stops being curve-fitting and starts being the language of causal-inference questions, so the exercises (and exam) lean that way. One small supplementary section at the end (bad controls / mediators) goes beyond the book's treatment, because it's the natural next question after confounding.

Ask for a hint if you get stuck; the first couple of asks per question get a nudge, not the answer. The exam at the bottom is the intuition check.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

pd.set_option("display.max_columns", None)

ames = pd.read_csv('../data/ames_housing.csv')
ames.shape

## 1. Correlated Predictors — When the Sign Flips

Two predictors that move together both "explain" the same variation in y; once one is in the model, the other only gets credit for what's *left over*. Numeric picture: houses with more bedrooms tend to be bigger, and bigger houses cost more. Looking at bedrooms alone, a bedroom appears to "add" a good chunk of value — because it's standing in for square footage. Hold square footage fixed and compare houses of the *same size*: the one with 4 bedrooms has smaller rooms than the one with 3, and buyers may like big rooms. The coefficient can shrink, vanish, or even flip sign.

**Question 1.** Fit these three models and compare the `Bedroom_AbvGr` coefficient: (i) `Sale_Price ~ Bedroom_AbvGr`; (ii) `... + Gr_Liv_Area`; (iii) `... + Gr_Liv_Area + TotRms_AbvGrd`. (a) What happened between (i) and (ii)? Compute `Gr_Liv_Area` average by number of bedrooms to explain the mechanism in numbers. (b) The coefficient in (ii) is *negative*. Is it right to conclude "bedrooms reduce house prices"? What is the comparison it actually describes? (c) What happens to `TotRms_AbvGrd` and the standard errors in (iii)? What does that tell you about the amount of independent information in those three variables?

*Tools for this question:*
- `df.groupby("Bedroom_AbvGr")["Gr_Liv_Area"].mean()`; read one coefficient with `res.params["Bedroom_AbvGr"]` and `res.bse["Bedroom_AbvGr"]`.

_Your interpretation:_

## 2. Multicollinearity

When predictors are highly correlated, the model can't tell their effects apart: many different coefficient combinations fit about equally well, so the estimates are **unstable** (huge standard errors, swing wildly between samples) even though the *predictions* may be fine. The **variance inflation factor (VIF)** quantifies this: regress predictor j on all the *other* predictors, take that regression's R², and compute `VIF = 1 / (1 - R²)`. If the other predictors explain 90% of predictor j, VIF = 1/(1-0.9) = 10, which means its coefficient's variance is ten times what it'd be if the predictors were uncorrelated (standard error ~3.2 times bigger).

The extreme case is **perfect multicollinearity**: one predictor is an exact sum of others. In Ames, `Gr_Liv_Area` equals `First_Flr_SF + Second_Flr_SF + Low_Qual_Fin_SF` exactly.

**Question 2.** (a) Verify the identity above for all 2,930 homes. Fit `Sale_Price ~ Gr_Liv_Area + First_Flr_SF + Second_Flr_SF + Low_Qual_Fin_SF` and read the warnings / `np.linalg.matrix_rank` of the design matrix. What happens and why? (b) Now take a *fixable* case: `Gr_Liv_Area, First_Flr_SF, Total_Bsmt_SF, Year_Built, Garage_Area`. Compute the VIF of each *by hand* via the auxiliary-regression recipe, then confirm with `variance_inflation_factor`. Which are inflated, and does it match your intuition about which variables overlap? (c) Bootstrap (500+ resamples): fit `Sale_Price ~ Gr_Liv_Area + First_Flr_SF + Total_Bsmt_SF` each time and record the coefficients and a prediction for a typical house. Compare the spread of the coefficients to the spread of the prediction. What does that tell you about when to worry about multicollinearity? (d) Name three things you might do about it and say which you'd pick if the goal is (i) prediction, (ii) explanation.

*Tools for this question:*
- `variance_inflation_factor(X.values, i)` (from `statsmodels.stats.outliers_influence`): `X` should be the predictor DataFrame **with** `sm.add_constant(...)`, and `i` is the column position to score.
- By hand: regress predictor j on the others, `1 / (1 - r2)`. Bootstrap: `df.sample(n=len(df), replace=True, random_state=i)`; `res.predict(new_df)`.

_Your interpretation:_

## 3. Confounding Variables

A **confounder** is something that drives both the variable you care about and the outcome, creating a misleading association. Numeric picture: suppose houses with central air sell for $60k more on average. But central air is much more common in newer houses, and old houses are cheap for lots of reasons unrelated to air conditioning. Compare *only houses of similar age and quality* and the AC premium shrinks — the raw gap mixed the effect of AC with the effect of everything AC is correlated with.

**Question 3.** Fit a sequence of models for `Sale_Price` with the `Central_Air` coefficient tracked at each step: (i) `Central_Air` only; (ii) + `Year_Built`; (iii) + `Gr_Liv_Area`; (iv) + numeric overall quality (map `Overall_Qual` to 1-10 — reuse your dictionary from the factor-variables notebook); (v) + `C(Neighborhood)`. (a) Tabulate the Central_Air coefficient and its standard error across the five models. What's happening to it, and why does each control matter? (b) At model (v), is the coefficient "the value of installing air conditioning in a house"? Name one more confounder you can't observe in this dataset, and say which direction it would bias the estimate. (c) What if the coefficient had *grown* after adding controls rather than shrunk — what would that imply about the sign of the relationship between the control and the treatment?

*Tools for this question:*
- A `C(Central_Air)` coefficient is named `C(Central_Air)[T.Y]`; collect results with `res.params[name]`, `res.bse[name]` in a list, then `pd.DataFrame`.

_Your interpretation:_

## 4. The Omitted-Variable-Bias Identity *(supplementary — not in the book)*

There's an exact accounting rule behind "controlling for X". Let the *long* model be `y ~ a + b*x + c*z` and the *short* model be `y ~ a' + b'*x` (leaving z out). Then **b' = b + c * δ**, where δ is the slope from regressing z on x. Numeric picture: suppose z = year built, x = central air, and in the long model AC is worth $13k and each year of age is worth $1,000 per year newer. If AC houses are on average 25 years newer than non-AC houses (δ = 25), the short model's AC coefficient is 13,000 + 1,000 × 25 = $38,000 — the direct effect plus the effect of the newer-age that tags along with AC. This is a mathematical identity of least squares, not an approximation.

**Question 4.** (a) For `x = Central_Air` (as a 0/1 column) and `z = Year_Built`, *predict the sign* of (b' − b) before computing. (b) Fit the three regressions needed (short, long, auxiliary `Year_Built ~ Central_Air`) and verify the identity numerically. (c) Do the same with `z = Gr_Liv_Area`. Does the omitted variable matter more for Central_Air through age or through size? (d) In your own words: if you wanted to know how big a hidden confounder would have to be to erase an observed effect, how would you use this identity?

*Tools for this question:*
- For a 0/1 treatment column: `(df["Central_Air"] == "Y").astype(int)`. Auxiliary regression: `smf.ols("Year_Built ~ Central_Air_01", data=df).fit().params`.

_Your interpretation:_

## 5. Interactions and Main Effects

An **interaction** lets one variable's effect depend on another's. Numeric picture: if price rises $50 per sqft for homes without central air and $110 per sqft with it, then "the effect of a square foot" isn't a single number: it's 50 for the first group and 110 for the second. The model `y ~ x * g` includes `x`, `g`, and the product `x:g`. The coefficient on `x` is now the slope **for the reference group of g only**, the coefficient on `g` is the gap **at x = 0**, and the product term is the *difference in slope* between groups.

**Question 5.** (a) Fit `Sale_Price ~ Gr_Liv_Area * C(Central_Air)`. Write out the fitted line for houses with and without central air as two equations with explicit slopes and intercepts. (b) The `Central_Air[T.Y]` coefficient is negative. Explain why that does not mean AC lowers price, in terms of what x = 0 means here. (c) Center `Gr_Liv_Area` at its mean and re-fit. What changed, what didn't (R², the interaction coefficient, predictions)? (d) Test whether the interaction improves the model with an F-test (`anova_lm` comparing the model with and without the product term). Is it a statistically *and* practically meaningful improvement? Look at the plot of the two fitted lines over the data. (e) Look at how many no-AC homes there are and how big they are. How well-determined is the no-AC slope likely to be, and what does the standard error say?

*Tools for this question:*
- `"y ~ x * C(g)"` expands to `x + C(g) + x:C(g)`. The product coefficient is named like `Gr_Liv_Area:C(Central_Air)[T.Y]`.
- `anova_lm(res_without, res_with)` (from `statsmodels.stats.anova`) gives the F-test for nested models. Centering: `df["x"] - df["x"].mean()`.

_Your interpretation:_

## 6. Supplementary: Controls That Hurt (Mediators)

*(Beyond the book — it only touches on confounding — but it's the natural follow-up.)* Not every variable correlated with both treatment and outcome should be controlled for. If a variable sits *between* the cause and the effect (a **mediator**), controlling for it blocks the very pathway you're trying to measure. Numeric picture: you ask whether a remodel raises price. A remodel works partly by raising overall quality, which buyers pay for. Control for quality and you're comparing remodeled and unremodeled homes *of the same quality* — which wipes out the main way remodeling helps.

**Question 6.** Data quirk first: `Year_Remod_Add` is floored at 1950 (any home built or remodeled before 1950 shows 1950), so for older homes `Year_Remod_Add > Year_Built` is true whether or not a remodel happened. Check this, then restrict to homes with `Year_Built >= 1950` and create `remod = (Year_Remod_Add > Year_Built)`. (a) Fit `Sale_Price ~ remod`, then `+ Year_Built + Gr_Liv_Area`, then `+ numeric overall quality`. Report the `remod` coefficient each time. (b) Regress overall quality (numeric) on `remod + Year_Built + Gr_Liv_Area`. What does the `remod` coefficient there tell you about quality as a mediator? (c) Which of the three remod coefficients would you report as "the effect of remodeling on price"? Which would you report as "the effect of remodeling *that doesn't work through overall quality*"? (d) Compare the raw coefficient to the one after adding `Year_Built`. Use the omitted-variable identity from Section 4 to explain whatever change you see.

*Tools for this question:*
- `(df["Year_Remod_Add"] > df["Year_Built"]).astype(int)` for the flag; `df.query("Year_Built >= 1950")` to restrict.

_Your interpretation:_

---
# Exam — check your intuition

7 questions. Almost none need code: the model outputs are printed for you as text, and you're asked to *read*, *predict*, or *explain*. Scratch code is fine if you want to poke at something, but try to answer from understanding first.

How it works:
- Write your answer in the **Your answer** cell under each question. Plain words, a few bullets — the way you'd say it out loud in an interview.
- Leave the **Review** cell alone. When you're done, ask Claude to review; it fills those cells in. Being right is a perfectly valid review outcome.
- Each question is tagged with the role it leans toward — **DS** (general data science), **Causal** (decision science / causal inference), **ML** (machine learning). Anything tagged for one role is still fair game for the others.
- Several questions are phrased the way interviewers ask them. A good answer is short, names the mechanism, and says what you'd *do* about it.

### Exam Q1 — Two correlated predictors in one model
*Role lean: DS · ML*

Garage capacity can be measured by number of cars or by square feet — two measures of nearly the same thing. Three models:

```text
corr(Garage_Cars, Garage_Area) = 0.89      VIF of each predictor in the two-variable model = 4.8

MODEL A: Sale_Price~Garage_Cars
                coef  std err     t   P>|t|
Intercept    60753.4   2842.6  21.4  <0.001
Garage_Cars  67966.2   1478.0  46.0  <0.001
   held-out RMSE (70/30 split) = 63,308

MODEL B: Sale_Price~Garage_Area
                coef  std err     t   P>|t|
Intercept    68470.4   2737.3  25.0  <0.001
Garage_Area    237.6      5.3  45.1  <0.001
   held-out RMSE (70/30 split) = 63,020

MODEL C: Sale_Price~Garage_Cars+Garage_Area
                coef  std err     t   P>|t|
Intercept    57526.7   2812.8  20.5  <0.001
Garage_Cars  39294.5   3185.1  12.3  <0.001
Garage_Area    114.0     11.3  10.1  <0.001
   held-out RMSE (70/30 split) = 61,932
```

(a) Both variables are strongly "significant" in Model C, yet the coefficients are about half of what they were alone. Say in plain words what the `Garage_Cars` coefficient of about 39,300 in Model C means versus the roughly 68,000 in Model A. (b) The standard error on `Garage_Area` more than doubled from Model B to Model C. Why? (c) Held-out RMSE barely moved across the three models. What does that tell you about when multicollinearity is and isn't a problem — for *prediction* versus *explaining which variable matters*? (d) A colleague says "VIF is only about 5, so there's no problem." What would you want to know before agreeing?

**Your answer (Q1):**

> _write here_

**Review (Q1):**

> _pending — filled in after you answer_

### Exam Q2 — Why the coefficients wobble but the prediction doesn't
*Role lean: DS · ML · interview*

Continuing with Model C, here is how much its numbers jump around if you re-fit on bootstrap resamples of the data:

```text
1,000 bootstrap re-fits of MODEL C (Sale_Price ~ Garage_Cars + Garage_Area):
  std dev of the Garage_Cars coefficient across re-fits : 4,056  (full-data estimate 39,295)
  std dev of the Garage_Area coefficient across re-fits : 15.9  (full-data estimate 114.0)
  correlation between the two coefficient estimates     : -0.87
  predicted price for a 2-car, 500 sq ft garage: mean 193,153, std dev across re-fits 1,397
```

(a) The two coefficient estimates are strongly *negatively* correlated across re-fits. Explain intuitively why — what has to happen to the Area coefficient when the Cars coefficient bounces up? (b) Why is the *predicted price* so much more stable than either coefficient? (c) What does this say about whether you'd report "the effect of one more car stall" from this model to a stakeholder? What would you do instead (name two options)?

**Your answer (Q2):**

> _write here_

**Review (Q2):**

> _pending — filled in after you answer_

### Exam Q3 — Predicting the sign of omitted-variable bias
*Role lean: Causal (supplementary) · DS · interview*

A short model leaves out `Year_Built`; a long model includes it. Everything below is from real fits:

```text
SHORT model: Sale_Price ~ Garage_Area
   Garage_Area coef = 237.6

LONG model:  Sale_Price ~ Garage_Area + Year_Built
   Garage_Area coef = 179.5     Year_Built coef = 861.2

Auxiliary:   Year_Built ~ Garage_Area
   Garage_Area coef = 0.0676   (each extra sq ft of garage goes with a house that is 0.0676 years newer, on average)
```

(a) Before looking at the numbers, which direction would you have predicted the bias to go (does the short-model coefficient on Garage_Area overstate or understate the long-model one), using only: "newer houses sell for more" and "newer houses have bigger garages"? (b) Verify the identity short = long + (Year_Built coef in long) × (Garage_Area coef in auxiliary) with the numbers above. (c) State the identity in words and use it to answer: if the omitted variable were *uncorrelated* with the one you kept, what would happen? If it were *negatively* correlated with the kept variable but positively related to price, which way would the bias go? (d) Why is this the formula behind most "controlling for X" conversations in causal inference?

**Your answer (Q3):**

> _write here_

**Review (Q3):**

> _pending — filled in after you answer_

### Exam Q4 — How many controls are enough?
*Role lean: Causal*

A real-estate agent wants to know what a fireplace is worth. Here is the coefficient on the number of fireplaces as controls are added:

```text
Coefficient on Fireplaces (std err), by what else is in the model:
  Fireplaces only                                              58,512  (2,006)
  + Gr_Liv_Area                                                23,792  (1,756)
  + Gr_Liv_Area + Year_Built + Neighborhood (28 levels)        16,721  (1,297)

Average price by number of fireplaces: 0: 141,196, 1: 213,556, 2: 242,316, 3: 255,821, 4: 260,000
```

(a) Tell the story of why the coefficient shrank from 58,512 to 16,721 — what are fireplaces a proxy for? (b) Does 16,721 mean "installing a fireplace in a house raises its sale price by about $16.7k"? Give two specific reasons it might not (one about variables not in the model, one about the kind of change being compared). (c) The coefficient didn't go to zero. Does that mean fireplaces cause higher prices? What would you do to gauge how fragile the 16.7k is? (hint: what would a hidden confounder have to look like to erase it?)

**Your answer (Q4):**

> _write here_

**Review (Q4):**

> _pending — filled in after you answer_

### Exam Q5 — A control that hurts
*Role lean: Causal (supplementary) · DS · interview*

An agent wants to know what *adding a full bathroom* is worth. Here is the coefficient on `Full_Bath` as controls are added:

```text
Coefficient on Full_Bath (std err), by what else is in the model:
  Full_Bath only                         78,827  (2,238)
  + Year_Built                           52,530  (2,313)
  + Year_Built + Gr_Liv_Area             -8,856  (2,208)

Auxiliary:  Gr_Liv_Area ~ Full_Bath + Year_Built  ->  coefficient on Full_Bath = 606 sq ft per extra full bath
Full_Bath counts: 0 baths: 12, 1 baths: 1318, 2 baths: 1532, 3 baths: 64, 4 baths: 4
```

(a) Walk through the three coefficients: why does adding `Year_Built` shrink the number, and why does the last step flip the sign? (b) The auxiliary regression says each extra full bath goes with about 606 more square feet of living area. In causal-inference language, what role does `Gr_Liv_Area` play if the question is "what is a bathroom *addition* worth" — and what question does the third coefficient actually answer instead? (c) Which of the three numbers would you report for the agent's question, and which for a question like "among homes of identical size, does swapping a bedroom for a second bath raise price"? (d) Interview version: "You're told to include every available covariate to be safe. What could go wrong?" Give an example from this table.

**Your answer (Q5):**

> _write here_

**Review (Q5):**

> _pending — filled in after you answer_

### Exam Q6 — Reading an interaction
*Role lean: DS · Causal · interview*

A model that lets the price-per-square-foot differ between single-family homes and end-unit townhouses:

```text
Sale_Price ~ Gr_Liv_Area * C(Bldg_Type), only OneFam and TwnhsE homes
(reference level = OneFam)
                                       coef  std err     t   P>|t|
Intercept                            6094.2   3419.2   1.8   0.075
C(Bldg_Type)[T.TwnhsE]             -27118.1  17321.1  -1.6   0.118
Gr_Liv_Area                           117.7      2.1  55.3  <0.001
Gr_Liv_Area:C(Bldg_Type)[T.TwnhsE]     41.2     12.5   3.3   0.001

n = 2658   R-squared = 0.549   Adj. R-squared = 0.548   Residual std error = 54,801
Average Gr_Liv_Area: OneFam 1,518, TwnhsE 1,342
```

(a) What is the estimated price per extra sq ft for a OneFam home? For a TwnhsE home? (b) The `C(Bldg_Type)[T.TwnhsE]` coefficient is -27,118. Does that mean townhouses sell for $27k less? Where does that number apply, and is that a place you have data? (c) Compute the predicted TwnhsE-minus-OneFam price gap at 1,000 sq ft and at 2,500 sq ft. At roughly what size do the two fitted lines cross? (d) How would centering `Gr_Liv_Area` at its mean change the dummy coefficient's meaning, and what would stay the same? (e) The interaction has p = 0.001 but the TwnhsE main effect has p = 0.118. A teammate says "drop the non-significant main effect." Do you agree?

**Your answer (Q6):**

> _write here_

**Review (Q6):**

> _pending — filled in after you answer_

### Exam Q7 — Choose your controls
*Role lean: Causal (supplementary) · DS*

You're estimating the effect on sale price of *upgrading a kitchen* (`Kitchen_Qual`) using Ames data. For each candidate variable, classify its role relative to that question — **confounder** (common cause of kitchen quality and price), **mediator** (lies on the path from kitchen to price), **collider/post-treatment variable** (caused by price or by both), or **irrelevant** — and say whether you'd control for it and why: (1) `Neighborhood`, (2) `Year_Built`, (3) `Overall_Qual`, (4) a hypothetical column "number of days the listing was on the market" (affected by price, and by how nice the house is), (5) `Mo_Sold` (month of sale). Then: if you can only run observational regression, what residual assumption do you have to make for your estimate to be causal, and how would you stress-test it? (This question goes beyond the textbook, which only briefly mentions confounding — but it's the standard conversation in decision-science interviews.)

**Your answer (Q7):**

> _write here_

**Review (Q7):**

> _pending — filled in after you answer_